# CS3081 - Exercise 5(c)
Abdulrahman Al Shalhoub (S24109270)

In [7]:
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD, State
from pgmpy.inference import VariableElimination
from pgmpy.sampling import BayesianModelSampling

model = DiscreteBayesianNetwork([
    ("heatwave", "overload"),
    ("heatwave", "powercut"),
    ("overload", "powercut"),
    ("powercut", "cancel")
])

heatwave = TabularCPD(
    variable="heatwave",
    variable_card=2,
    values=[[0.3], [0.7]],
    state_names={"heatwave": ["yes", "no"]}
)

overload = TabularCPD(
    variable="overload",
    variable_card=2,
    values=[[0.6, 0.1],
            [0.4, 0.9]],
    evidence=["heatwave"],
    evidence_card=[2],
    state_names={"overload": ["yes", "no"],
                 "heatwave": ["yes", "no"]}
)

powercut = TabularCPD(
    variable="powercut",
    variable_card=2,
    values=[[0.7, 0.2, 0.5, 0.05],
            [0.3, 0.8, 0.5, 0.95]],
    evidence=["heatwave", "overload"],
    evidence_card=[2, 2],
    state_names={"powercut": ["yes", "no"],
                 "heatwave": ["yes", "no"],
                 "overload": ["yes", "no"]}
)

cancel = TabularCPD(
    variable="cancel",
    variable_card=2,
    values=[[0.8, 0.1],
            [0.2, 0.9]],
    evidence=["powercut"],
    evidence_card=[2],
    state_names={"cancel": ["yes", "no"],
                 "powercut": ["yes", "no"]}
)

model.add_cpds(heatwave, overload, powercut, cancel)
print(model.check_model())


True


In [8]:
inference = VariableElimination(model)
result = inference.query(["heatwave"], evidence={"cancel": "yes"}, show_progress=False)
exact_yes = result.get_value(heatwave="yes")
exact_no = result.get_value(heatwave="no")
print(f"P(heatwave = yes | cancel = yes) = {exact_yes:.4f}")
print(f"P(heatwave = no  | cancel = yes) = {exact_no:.4f}")


P(heatwave = yes | cancel = yes) = 0.5367
P(heatwave = no  | cancel = yes) = 0.4633


In [9]:
sampler = BayesianModelSampling(model)
samples = sampler.rejection_sample(
    evidence=[State("cancel", "yes")], size=10000, seed=42, show_progress=False
)
heatwave_count = (samples["heatwave"] == "yes").sum()
estimate = heatwave_count / len(samples)
print(f"Heatwave = yes in {heatwave_count:,} out of {len(samples):,} kept samples.")
print(f"Rejection sampling estimate = {estimate:.4f}")


Heatwave = yes in 5,430 out of 10,000 kept samples.
Rejection sampling estimate = 0.5430


### Comparison
The exact answer was 0.5367, the same as in Exercise 4. 
The sample estimate was 0.5430, and it can change with a different set of samples.